<a href="https://colab.research.google.com/github/manijawahar/python/blob/main/Myfirstgmail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import os
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# If modifying these scopes, delete the file token.json.
# We need read-only access to emails
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

def authenticate_gmail():
    """Authenticates with the Gmail API and returns the service object."""
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json", SCOPES
            )
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    try:
        service = build("gmail", "v1", credentials=creds)
        return service
    except HttpError as error:
        print(f"An error occurred during authentication: {error}")
        return None

def list_latest_emails(service, num_emails=10):
    """Lists the latest emails from the user's inbox."""
    if not service:
        print("Gmail service not available.")
        return

    try:
        # Get the latest emails from the inbox
        results = service.users().messages().list(userId='me', labelIds=['INBOX'], maxResults=num_emails).execute()
        messages = results.get('messages', [])

        if not messages:
            print("No new emails found.")
        else:
            print(f"Latest {num_emails} emails:")
            for message in messages:
                msg = service.users().messages().get(userId='me', id=message['id']).execute()
                # You can parse the message payload to get subject, sender, etc.
                # For simplicity, we'll just print the message ID here
                print(f"  - Message ID: {msg['id']}")
                # You can uncomment the following lines to see the full message payload
                # import json
                # print(json.dumps(msg, indent=2))

    except HttpError as error:
        print(f"An error occurred while fetching emails: {error}")

if __name__ == "__main__":
    gmail_service = authenticate_gmail()
    if gmail_service:
        list_latest_emails(gmail_service)

FileNotFoundError: [Errno 2] No such file or directory: 'credentials.json'